# Optional evaluation

Run the setup and definitions below. Uncomment the desired call in the final cell. Field CSV columns are image_path, garment_id and class_id. Relative image paths resolve from the CSV location.

In [ ]:
from pathlib import Path
import os

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "Notebooks/01_dataset_inventory.ipynb").is_file())
os.chdir(root)

%run Notebooks/05_classification_setup.ipynb

In [ ]:
from hashlib import sha256

def evaluate_checkpoint(checkpoint_path, architecture, variant, batch_size=32):
    checkpoint_path = Path(checkpoint_path)
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    net = build_model(architecture, dropout=0, pretrained=False)
    net.load_state_dict(checkpoint["model_state_dict"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net.to(device)
    frame = load_manifest()
    digest = sha256(checkpoint_path.read_bytes()).hexdigest()[:16]
    folder = ROOT / "reports/model_evaluation/checkpoints" / architecture / variant / digest
    for split in ("validation", "test"):
        dataset = SeamImages(frame.loc[frame.split == split], variant)
        evaluate(net, DataLoader(dataset, batch_size=batch_size), device, folder / split)
    print(folder)

In [ ]:
def compare(manifest, output):
    photos = pd.read_csv(manifest)
    if photos.empty or not {"image_path", "garment_id", "class_id"} <= set(photos):
        raise ValueError("Provide image_path, garment_id and class_id for each labeled photo.")
    if photos[["image_path", "garment_id", "class_id"]].isna().any().any() or not photos.class_id.isin(range(11)).all():
        raise ValueError("Each photo needs a garment ID and an integer class ID from 0 to 10.")
    paths = [(manifest.parent / name).resolve() for name in photos.image_path]
    if len(set(paths)) != len(paths):
        raise ValueError("Duplicate photo paths in field manifest.")
    checkpoints = {
        name: ROOT / "Notebooks/models/exp18_resnet18_fpn" / f"{name}_best.pth"
        for name in ("resnet18_baseline", "resnet18_fpn")
    }
    for name in ("efficientnet_b0_fpn", "densenet121_fpn", "convnext_tiny_fpn"):
        checkpoints[name] = ROOT / "models/experiment19_fpn_backbones" / name / "best_model.pth"
    output.mkdir(parents=True, exist_ok=True)
    summaries = []
    for name, path in checkpoints.items():
        checkpoint = torch.load(path, map_location="cpu", weights_only=True)
        model = build_model(name, dropout=0, pretrained=False).eval()
        model.load_state_dict(checkpoint["model_state_dict"])
        transform = evaluation_transform()
        records = []
        with torch.inference_mode():
            for image_path in paths:
                with Image.open(image_path) as source:
                    image = transform(ImageOps.exif_transpose(source).convert("RGB"))
                score, label = model(image.unsqueeze(0)).softmax(1)[0].max(0)
                records.append((label.item(), score.item()))
        predictions = photos.copy()
        predictions[["predicted_id", "score"]] = records
        predictions["predicted_id"] = predictions.predicted_id.astype(int)
        predictions["correct"] = predictions.class_id == predictions.predicted_id
        predictions.to_csv(output / f"{name}_predictions.csv", index=False)
        classes = predictions.groupby("class_id").correct.agg(images="size", recall="mean")
        classes.to_csv(output / f"{name}_per_class.csv")
        summaries.append(dict(model=name, images=len(photos), garments=photos.garment_id.nunique(),
                              wrong=int((~predictions.correct).sum()),
                              defects_called_normal=int(((photos.class_id != 0) & (predictions.predicted_id == 0)).sum()),
                              confident_errors=int(((~predictions.correct) & (predictions.score >= 0.9)).sum()),
                              macro_f1_on_present_classes=f1_score(photos.class_id, predictions.predicted_id,
                                  labels=sorted(photos.class_id.unique()), average="macro", zero_division=0)))
    result = pd.DataFrame(summaries)
    result.to_csv(output / "comparison.csv", index=False)
    return result

In [ ]:
def rebuild_historical_reports():
    REPORTS = ROOT / "reports/model_evaluation"
    OUTPUT = REPORTS / "deployment_selection"
    RGB = REPORTS / "experiment19_fpn_backbones"
    ENHANCED = REPORTS / "experiment20_enhanced_fpn"
    CHOSEN = "efficientnet_b0_fpn"
    OUTPUT.mkdir(exist_ok=True)
    reference = pd.read_csv(RGB / f"{CHOSEN}_validation_predictions.csv").set_index("sample_id").class_id.sort_index()
    assert reference.index.is_unique
    rows = []
    for directory, names in [(RGB, ["efficientnet_b0_fpn", "convnext_tiny_fpn", "densenet121_fpn"]),
                             (ENHANCED, ["lab_clahe", "bilateral_clahe"])]:
        for name in names:
            path = directory / f"{name}_validation_predictions.csv"
            df = pd.read_csv(path)
            assert df.set_index("sample_id").class_id.sort_index().equals(reference)
            rows.append(dict(model=name if directory == RGB else CHOSEN,
                             input="original_rgb" if directory == RGB else name,
                             macro_f1=f1_score(df.class_id, df.predicted_class_id, average="macro"),
                             wrong=int((df.class_id != df.predicted_class_id).sum()), images=len(df),
                             source=str(path.relative_to(ROOT))))
    for name in ["resnet18_baseline", "resnet18_fpn"]:
        path = ROOT / f"Notebooks/models/exp18_resnet18_fpn/{name}_best.pth"
        metrics = torch.load(path, map_location="cpu", weights_only=True)["validation_metrics"]
        rows.append(dict(model=name, input="original_rgb", macro_f1=metrics["macro_f1"],
                         wrong=round(len(reference)*(1-metrics["accuracy"])), images=len(reference),
                         source=str(path.relative_to(ROOT))))
    ranking = pd.DataFrame(rows).sort_values("macro_f1", ascending=False)
    ranking.to_csv(OUTPUT / "validation_leaderboard.csv", index=False)
    if ranking.iloc[0].model != CHOSEN or ranking.iloc[0].input != "original_rgb":
        raise ValueError("Review the new winner before updating deployment.")
    for split in ["validation", "test"]:
        df = pd.read_csv(RGB / f"{CHOSEN}_{split}_predictions.csv").rename(
            columns={"folder_1": "fabric", "folder_2": "actual_class"})
        df["correct"] = df.class_id == df.predicted_class_id
        folder = OUTPUT / CHOSEN / "original_rgb" / split
        folder.mkdir(parents=True, exist_ok=True)
        predictions = df[["sample_id", "s3_key", "fabric", "actual_class", "predicted_class", "confidence", "correct"]]
        predictions.sort_values("confidence").to_csv(folder / "all_predictions.csv", index=False)
        predictions.loc[~df.correct].sort_values("confidence", ascending=False).to_csv(folder / "wrong_predictions.csv", index=False)
        for group in ["fabric", "actual_class"]:
            summary = df.groupby(group).correct.agg(images="size", correct="sum")
            summary["wrong"] = summary.images - summary.correct
            summary["recall" if group == "actual_class" else "accuracy"] = summary.correct / summary.images
            summary.to_csv(folder / f"performance_by_{group}.csv")
        print(split, len(df), "errors", (~df.correct).sum(), "defects called normal", ((df.class_id != 0) & (df.predicted_class_id == 0)).sum())

In [ ]:
# rebuild_historical_reports()
# evaluate_checkpoint("models/experiment19_fpn_backbones/efficientnet_b0_fpn/best_model.pth", "efficientnet_b0_fpn", "original_rgb")
# display(compare(Path("field_labels.csv"), root / "reports/field_evaluation"))